<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public class Customer
{
    protected string _email;

    public int CustomerId { get; protected set; }
    public string Name { get; protected set; }
    public string Email
    {
        get => _email;
        set
        {
            if (string.IsNullOrWhiteSpace(value) || !value.Contains("@"))
                throw new ArgumentException("Некорректный email.");
            _email = value;
        }
    }

    public Customer(int customerId, string name, string email)
    {
        if (customerId <= 0) throw new ArgumentException("ID должен быть положительным.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Имя не может быть пустым.");
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName()
    {
        return Name;
    }

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        Console.WriteLine($"📧 Email обновлён: {Email}");
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine($"👤 ID: {CustomerId} | Имя: {GetFullName()} | Email: {Email}");
    }

    // Перегрузка метода ViewProfile
    public virtual void ViewProfile(bool detailed)
    {
        if (detailed)
        {
            Console.WriteLine("🔍 Подробный профиль:");
            ViewProfile();
        }
        else
        {
            Console.WriteLine($"[Кратко] {GetFullName()} <{Email}>");
        }
    }
}

public class VipCustomer : Customer
{
    public int LoyaltyPoints { get; private set; }
    public string Tier { get; private set; }
    public DateTime VipSince { get; private set; }
    public bool HasDedicatedManager { get; private set; }

    public VipCustomer(int customerId, string name, string email, int loyaltyPoints)
        : base(customerId, name, email)
    {
        if (loyaltyPoints < 0) throw new ArgumentException("Баллы не могут быть отрицательными.");
        LoyaltyPoints = loyaltyPoints;
        VipSince = DateTime.Now.AddYears(-1);
        Tier = loyaltyPoints >= 1000 ? "Platinum" : "Gold";
        HasDedicatedManager = loyaltyPoints >= 500;
    }

    public void AddLoyaltyPoints(int points)
    {
        LoyaltyPoints += points;
        Console.WriteLine($"🎁 +{points} баллов. Всего: {LoyaltyPoints}");
    }

    public void RequestPrioritySupport()
    {
        Console.WriteLine($"{Name} запрашивает приоритетную поддержку!");
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"🌟 [VIP] ID: {CustomerId} | Имя: {Name} | Email: {Email}");
        Console.WriteLine($"   💎 Баллы: {LoyaltyPoints} | Уровень: {Tier} | Менеджер: {(HasDedicatedManager ? "Да" : "Нет")}");
    }

    public override void ViewProfile(bool detailed)
    {
        if (detailed)
        {
            Console.WriteLine("💎 Подробный VIP-профиль:");
            ViewProfile();
        }
        else
        {
            Console.WriteLine($"[VIP] {Name} ({Tier})");
        }
    }
}

public class RegularCustomer : Customer
{
    public DateTime RegistrationDate { get; protected set; }
    public DateTime? LastEmailUpdate { get; private set; }
    public int PurchaseCount { get; set; }
    public string FavoriteCategory { get; set; } = "Общее";

    public RegularCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
    }

    public void MakePurchase(decimal amount)
    {
        PurchaseCount++;
        Console.WriteLine($"{Name} совершил покупку на {amount:C}. Всего: {PurchaseCount} покупок.");
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
        Console.WriteLine($"📅 Последнее обновление email: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"👤 [Обычный] ID: {CustomerId} | Имя: {Name} | Email: {Email}");
        Console.WriteLine($"   🗓 Регистрация: {RegistrationDate:yyyy-MM-dd} | Покупок: {PurchaseCount} | Категория: {FavoriteCategory}");
        if (LastEmailUpdate.HasValue)
            Console.WriteLine($"   📬 Последнее обновление email: {LastEmailUpdate:yyyy-MM-dd HH:mm}");
    }

    public override void ViewProfile(bool detailed)
    {
        if (detailed)
        {
            Console.WriteLine("🛒 Подробный профиль обычного клиента:");
            ViewProfile();
        }
        else
        {
            Console.WriteLine($"[Обычный] {Name} (с {RegistrationDate:yyyy})");
        }
    }
}

public class GroupCustomer : Customer
{
    public string GroupName { get; protected set; }
    public DateTime CreationDate { get; private set; }
    public int MaxMembers { get; private set; } = 50;
    private List<Customer> _members = new List<Customer>();

    public IReadOnlyList<Customer> Members => _members.AsReadOnly();

    public GroupCustomer(int customerId, string groupName, string email)
        : base(customerId, groupName, email)
    {
        GroupName = groupName;
        CreationDate = DateTime.Now;
    }

    public override string GetFullName()
    {
        return $"Группа «{GroupName}»";
    }

    public void AddMember(Customer member)
    {
        if (member == null) throw new ArgumentNullException(nameof(member));
        if (_members.Count >= MaxMembers)
        {
            Console.WriteLine($"❌ Группа '{GroupName}' полна ({MaxMembers} участников).");
            return;
        }
        if (_members.Contains(member))
        {
            Console.WriteLine($"⚠️ {member.Name} уже в группе.");
            return;
        }
        _members.Add(member);
        Console.WriteLine($"✅ {member.Name} добавлен в группу '{GroupName}'");
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"👥 [Группа] ID: {CustomerId} | Название: {GroupName} | Email: {Email}");
        Console.WriteLine($"   📅 Создана: {CreationDate:yyyy-MM-dd} | Участников: {_members.Count}/{MaxMembers}");
        if (_members.Any())
        {
            Console.WriteLine("   Участники:");
            foreach (var m in _members)
                Console.WriteLine($"     → {m.Name} (ID: {m.CustomerId})");
        }
    }

    public override void ViewProfile(bool detailed)
    {
        if (detailed)
        {
            Console.WriteLine("👥 Подробный профиль группы:");
            ViewProfile();
        }
        else
        {
            Console.WriteLine($"[Группа] «{GroupName}» ({_members.Count} участников)");
        }
    }
}

// Generic-класс для хранения клиентов
public class CustomerRepository<T> where T : Customer
{
    private List<T> _customers = new List<T>();

    public void Add(T customer)
    {
        _customers.Add(customer);
    }

    public void DisplayAll()
    {
        foreach (var c in _customers)
        {
            c.ViewProfile();
        }
    }

    public void DisplayAllBrief()
    {
        foreach (var c in _customers)
        {
            c.ViewProfile(false); // используем перегруженный метод
        }
    }

    public int Count => _customers.Count;
}

// === ДЕМОНСТРАЦИЯ ===
var vip = new VipCustomer(1, "Алексей", "alex@vip.com", 800);
var regular = new RegularCustomer(2, "Мария", "maria@test.com", new DateTime(2023, 5, 12));
var group = new GroupCustomer(100, "Команда Alpha", "alpha@org.com");

group.AddMember(vip);
group.AddMember(regular);

// Полиморфизм: массив базового типа
Customer[] all = { vip, regular, group };
Console.WriteLine("📋 Полиморфный вывод через базовый тип:");
foreach (var c in all)
{
    c.ViewProfile();
    Console.WriteLine();
}

// Generic-репозитории
var vips = new CustomerRepository<VipCustomer>();
vips.Add(vip);
vips.Add(new VipCustomer(5, "Елена", "elena@vip.com", 1200));

var regulars = new CustomerRepository<RegularCustomer>();
regulars.Add(regular);
regulars.Add(new RegularCustomer(3, "Иван", "ivan@test.com", new DateTime(2024, 1, 10)));

Console.WriteLine("\n📦 VIP-клиенты (кратко):");
vips.DisplayAllBrief();

Console.WriteLine("\n📦 Обычные клиенты (подробно):");
regulars.DisplayAll();

Console.WriteLine($"\n📊 Статистика: {vips.Count} VIP, {regulars.Count} обычных.");

✅ Алексей добавлен в группу 'Команда Alpha'
✅ Мария добавлен в группу 'Команда Alpha'
📋 Полиморфный вывод через базовый тип:
🌟 [VIP] ID: 1 | Имя: Алексей | Email: alex@vip.com
   💎 Баллы: 800 | Уровень: Gold | Менеджер: Да

👤 [Обычный] ID: 2 | Имя: Мария | Email: maria@test.com
   🗓 Регистрация: 2023-05-12 | Покупок: 0 | Категория: Общее

👥 [Группа] ID: 100 | Название: Команда Alpha | Email: alpha@org.com
   📅 Создана: 2025-11-03 | Участников: 2/50
   Участники:
     → Алексей (ID: 1)
     → Мария (ID: 2)


📦 VIP-клиенты (кратко):
[VIP] Алексей (Gold)
[VIP] Елена (Platinum)

📦 Обычные клиенты (подробно):
👤 [Обычный] ID: 2 | Имя: Мария | Email: maria@test.com
   🗓 Регистрация: 2023-05-12 | Покупок: 0 | Категория: Общее
👤 [Обычный] ID: 3 | Имя: Иван | Email: ivan@test.com
   🗓 Регистрация: 2024-01-10 | Покупок: 0 | Категория: Общее

📊 Статистика: 2 VIP, 2 обычных.
